# 🛰️ GPS Track Extractor — GPSMapCamera MP4

**What this notebook does:**  
Extracts continuous GPS coordinates (lat/lon) burned as overlay text into a GPSMapCamera video.  
It uses `ffmpeg` to pull frames, `Tesseract OCR` to read the overlay, and exports everything to CSV + an interactive map.

**Video used:** `Eng_clg_60504PMByGPSMapCamera.mp4`  
**Location:** Lucknow, Uttar Pradesh, India  
**Date:** 20 May 2026, ~6:05 PM IST

---
## Cell 1 — Install System Dependencies

> Run this cell **once**. It installs Tesseract OCR (the engine that reads text from images) and ffmpeg (the tool that extracts video frames).  
> - **Ubuntu/Debian:** runs `apt-get` commands below  
> - **macOS:** use `brew install tesseract ffmpeg` in Terminal instead  
> - **Windows:** download installers from https://github.com/UB-Mannheim/tesseract/wiki and https://ffmpeg.org/download.html

In [ ]:
import subprocess, sys

# Install Tesseract OCR engine (system-level, Ubuntu/Debian/Colab)
subprocess.run(['apt-get', 'install', '-y', 'tesseract-ocr', 'ffmpeg'], 
               capture_output=True, text=True)

# Verify both tools are available
for tool in ['tesseract', 'ffmpeg', 'ffprobe']:
    result = subprocess.run(['which', tool], capture_output=True, text=True)
    status = '✅' if result.returncode == 0 else '❌'
    print(f"{status} {tool}: {result.stdout.strip() or 'NOT FOUND'}")

---
## Cell 2 — Install Python Packages

> Installs all required Python libraries:  
> - `pytesseract` — Python wrapper for Tesseract OCR  
> - `Pillow` — image loading and enhancement  
> - `opencv-python-headless` — video frame reading  
> - `pandas`, `numpy` — data processing and interpolation  
> - `folium` — interactive HTML map  
> - `matplotlib` — plotting charts  
> - `tqdm` — progress bars

In [ ]:
import subprocess, sys

packages = [
    'pytesseract',
    'Pillow',
    'opencv-python-headless',
    'pandas',
    'numpy',
    'folium',
    'matplotlib',
    'tqdm'
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f"{status} {pkg}")

---
## Cell 3 — Import Libraries

> Import everything we need. If any import fails, re-run Cell 2.

In [ ]:
import os
import re
import subprocess
import numpy as np
import pandas as pd
import pytesseract
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import folium
import warnings

from PIL import Image, ImageEnhance
from math import radians, cos, sin, asin, sqrt
from tqdm.notebook import tqdm
from IPython.display import display, HTML, IFrame

warnings.filterwarnings('ignore')
print('✅ All libraries imported successfully')

---
## Cell 4 — Configuration

> **Edit `VIDEO_PATH` to point to your video file.**  
> All other settings have sensible defaults — change them only if needed.

| Variable | Default | Description |
|---|---|---|
| `VIDEO_PATH` | `'Eng_clg_60504PMByGPSMapCamera.mp4'` | Path to your GPSMapCamera video |
| `OUTPUT_CSV` | `'gps_track.csv'` | Where to save the extracted GPS data |
| `FRAMES_DIR` | `'frames_temp'` | Temporary folder for extracted frames |
| `FPS_EXTRACT` | `1` | Frames per second to extract (1 = one per second) |
| `CROP_BOTTOM_THIRD` | `True` | Crop to bottom third where GPS overlay appears |

In [ ]:
# ─── EDIT THIS ──────────────────────────────────────────────────────────
VIDEO_PATH        = 'Eng_clg_60504PMByGPSMapCamera.mp4'
# ────────────────────────────────────────────────────────────────────────

OUTPUT_CSV        = 'gps_track.csv'
FRAMES_DIR        = 'frames_temp'
FPS_EXTRACT       = 1       # 1 frame per second  (increase for finer resolution)
CROP_BOTTOM_THIRD = True    # GPSMapCamera overlay is in the bottom ~33% of frame

# Create frames folder
os.makedirs(FRAMES_DIR, exist_ok=True)

# Sanity check
if os.path.exists(VIDEO_PATH):
    size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024
    print(f'✅ Video found: {VIDEO_PATH}  ({size_mb:.1f} MB)')
else:
    print(f'❌ Video NOT found at: {VIDEO_PATH}')
    print('   → Update VIDEO_PATH above and re-run this cell.')

---
## Cell 5 — Inspect Video Metadata

> Uses `ffprobe` to read the video's resolution, frame rate, and duration.  
> This tells us exactly how many frames we'll extract.

In [ ]:
import cv2

cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
native_fps   = cap.get(cv2.CAP_PROP_FPS)
duration_s   = total_frames / native_fps
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.release()

expected_frames = int(duration_s * FPS_EXTRACT)

print('━' * 45)
print(f'  Resolution   : {width} × {height} px')
print(f'  Native FPS   : {native_fps:.2f}')
print(f'  Total frames : {total_frames:,}')
print(f'  Duration     : {duration_s:.1f} s  ({duration_s/60:.2f} min)')
print('━' * 45)
print(f'  Extract rate : {FPS_EXTRACT} fps')
print(f'  → ~{expected_frames} frames will be extracted')
print('━' * 45)

---
## Cell 6 — Extract Frames with ffmpeg

> Pulls one frame per second from the video.  
> The `-vf` (video filter) crops only the **bottom third** of each frame — that's where GPSMapCamera draws its overlay (lat, lon, address, timestamp).  
> Cropping massively speeds up OCR because Tesseract processes a much smaller image.

In [ ]:
# Build the ffmpeg filter:
#   fps={FPS_EXTRACT}          → keep only N frames per second
#   crop=iw:ih/3:0:2*ih/3      → crop to bottom third: full width, 1/3 height, x=0, y=2/3 down

vf_filter = f'fps={FPS_EXTRACT},crop=iw:ih/3:0:2*ih/3'

cmd = [
    'ffmpeg',
    '-i', VIDEO_PATH,
    '-vf', vf_filter,
    os.path.join(FRAMES_DIR, 'frame_%04d.jpg'),
    '-y',                 # overwrite if exists
    '-loglevel', 'error'  # suppress verbose output
]

print(f'⏳ Running ffmpeg...')
print(f'   Filter: {vf_filter}')

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print('❌ ffmpeg failed:')
    print(result.stderr)
else:
    extracted = len([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
    print(f'✅ Extracted {extracted} frames → {FRAMES_DIR}/')

---
## Cell 7 — Preview a Sample Frame

> Shows the first extracted frame so you can verify the GPS overlay is visible.  
> You should see something like: **Lat 26.913136° Long 80.939192°**  
> If the overlay is not in the bottom third, adjust the `crop` filter in Cell 6.

In [ ]:
frame_files = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])

# Show first, middle, and last frame
sample_indices = [0, len(frame_files)//2, -1]
sample_labels  = ['First frame (t=1s)', 'Middle frame', 'Last frame']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#0d0f12')

for ax, idx, label in zip(axes, sample_indices, sample_labels):
    img = Image.open(os.path.join(FRAMES_DIR, frame_files[idx]))
    ax.imshow(img)
    ax.set_title(label, color='white', fontsize=10, pad=6)
    ax.axis('off')

plt.suptitle('Extracted frames — GPS overlay region (bottom third)', 
             color='white', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()
print('→ Confirm you can see: Lat XX.XXXXXX° Long XX.XXXXXX° in the overlay text')

---
## Cell 8 — OCR: Extract GPS from Every Frame

> This is the core extraction loop. For each frame:  
> 1. Load the cropped image  
> 2. Boost contrast (makes white overlay text sharper for OCR)  
> 3. Convert to grayscale  
> 4. Run Tesseract with `--psm 6` (uniform block of text) and `--oem 3` (LSTM engine)  
> 5. Apply regex to find `Lat XX.XXXXXX° Long XX.XXXXXX°`  
> 6. Also capture the timestamp from the overlay  

**Expected accuracy: ~95%+** — a few frames may be blurry or occluded; those get interpolated in the next cell.

In [ ]:
# ── Regex patterns ──────────────────────────────────────────────────────
# Matches: "Lat 26.913136° Long 80.939192°" (GPSMapCamera format)
LAT_LON_RE = re.compile(
    r'[Ll]at\s*[:\-]?\s*([+\-]?\d{1,3}[.,]\d{4,})[°]?'
    r'\s*[Ll]o?n?g?\s*[:\-]?\s*([+\-]?\d{1,3}[.,]\d{4,})[°]?'
)
# Fallback: two decimal numbers close together (handles OCR misreads of "Lat"/"Long" prefix)
FALLBACK_RE = re.compile(r'(\d{2}\.\d{4,})[°\s]+?([78]\d\.\d{4,})')

# Matches: "Wednesday, 20/05/2026 06:05:09 PM GMT +05:30"
DT_RE = re.compile(
    r'(?:\w+,\s*)?(\d{1,2}/\d{2}/\d{4}\s+\d{1,2}:\d{2}:\d{2}\s*[AP]M(?:\s*GMT\s*[+\-]\d{2}:\d{2})?)'
    , re.IGNORECASE
)
# ────────────────────────────────────────────────────────────────────────

records = []

for i, fname in enumerate(tqdm(frame_files, desc='🔍 OCR frames')):
    path   = os.path.join(FRAMES_DIR, fname)
    second = i + 1

    # ── 1. Load image
    img = Image.open(path)

    # ── 2. Enhance: boost contrast so white text pops on map background
    img = ImageEnhance.Contrast(img).enhance(2.0)

    # ── 3. Grayscale (Tesseract works best on single-channel images)
    img = img.convert('L')

    # ── 4. OCR
    #   --psm 6  = assume uniform block of text
    #   --oem 3  = use LSTM neural net engine
    text = pytesseract.image_to_string(img, config='--psm 6 --oem 3')

    # ── 5. Parse lat/lon
    lat, lon = None, None
    m = LAT_LON_RE.search(text)
    if m:
        lat = float(m.group(1).replace(',', '.'))
        lon = float(m.group(2).replace(',', '.'))
    else:
        m2 = FALLBACK_RE.search(text)
        if m2:
            lat = float(m2.group(1))
            lon = float(m2.group(2))

    # ── 6. Parse datetime
    dt_str = ''
    dm = DT_RE.search(text)
    if dm:
        dt_str = dm.group(1).strip()

    records.append({
        'second'       : second,
        'latitude'     : lat,
        'longitude'    : lon,
        'datetime_IST' : dt_str
    })

df_raw = pd.DataFrame(records)
ok  = df_raw['latitude'].notna().sum()
bad = df_raw['latitude'].isna().sum()
print(f'\n✅ OCR complete')
print(f'   Successful readings : {ok}/{len(df_raw)} ({ok/len(df_raw)*100:.1f}%)')
print(f'   Failed (will interpolate): {bad}')
df_raw.head(5)

---
## Cell 9 — Clean Data & Interpolate Missing Values

> Some frames may fail OCR due to motion blur or partial occlusion.  
> Steps:  
> 1. Remove obvious outliers (coordinates outside the expected region)  
> 2. Fill gaps using **linear interpolation** (straight-line estimate between valid neighbours)  
> 3. Round to 6 decimal places (~0.1 m precision)

In [ ]:
df = df_raw.copy()

# ── Step 1: Force numeric (coerce bad OCR text to NaN)
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# ── Step 2: Remove outliers
# Adjust these bounds if your video is from a different city
LAT_MIN, LAT_MAX = 26.0, 28.0   # Lucknow latitude band
LON_MIN, LON_MAX = 80.0, 82.0   # Lucknow longitude band

df.loc[(df['latitude']  < LAT_MIN) | (df['latitude']  > LAT_MAX), 'latitude']  = np.nan
df.loc[(df['longitude'] < LON_MIN) | (df['longitude'] > LON_MAX), 'longitude'] = np.nan

# ── Step 3: Linear interpolation for missing values
n_missing_before = df['latitude'].isna().sum()
df['latitude']   = df['latitude'].interpolate(method='linear')
df['longitude']  = df['longitude'].interpolate(method='linear')
n_missing_after  = df['latitude'].isna().sum()

# ── Step 4: Round to 6 decimal places
df['latitude']  = df['latitude'].round(6)
df['longitude'] = df['longitude'].round(6)

print('Cleaning complete')
print(f'  Missing before interpolation : {n_missing_before}')
print(f'  Missing after  interpolation : {n_missing_after}')
print(f'  Final rows                   : {len(df)}')
print()
print('Coordinate range:')
print(f'  Lat: {df["latitude"].min():.6f}  →  {df["latitude"].max():.6f}')
print(f'  Lon: {df["longitude"].min():.6f}  →  {df["longitude"].max():.6f}')

---
## Cell 10 — Calculate Distance & Speed

> Uses the **Haversine formula** to compute great-circle distance between consecutive GPS points.  
> Adds two new columns:  
> - `segment_dist_m` — distance from the previous point (metres)  
> - `cumulative_dist_m` — running total distance from start (metres)

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    """Returns great-circle distance in metres between two WGS-84 coordinates."""
    R = 6_371_000  # Earth radius in metres
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * R * asin(sqrt(a))

# Calculate segment-by-segment distances
seg_dists = [0.0]  # first point has 0 distance
for i in range(1, len(df)):
    d = haversine_m(
        df['latitude'].iloc[i - 1], df['longitude'].iloc[i - 1],
        df['latitude'].iloc[i],     df['longitude'].iloc[i]
    )
    seg_dists.append(round(d, 3))

df['segment_dist_m']    = seg_dists
df['cumulative_dist_m'] = df['segment_dist_m'].cumsum().round(1)

total_m   = df['cumulative_dist_m'].iloc[-1]
total_km  = total_m / 1000
duration  = len(df)          # seconds
avg_kmh   = (total_km / duration) * 3600
max_seg   = df['segment_dist_m'].max()

print('━' * 40)
print(f'  Total distance : {total_m:.1f} m  ({total_km:.3f} km)')
print(f'  Duration       : {duration} seconds')
print(f'  Avg speed      : {avg_kmh:.1f} km/h')
print(f'  Max step dist  : {max_seg:.2f} m')
print('━' * 40)

---
## Cell 11 — Save to CSV

> Saves the final cleaned dataset.  
> Columns: `second`, `latitude`, `longitude`, `datetime_IST`, `segment_dist_m`, `cumulative_dist_m`

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)

print(f'✅ Saved: {OUTPUT_CSV}')
print(f'   Rows    : {len(df)}')
print(f'   Columns : {list(df.columns)}')
print()

# Pretty-print first 10 rows
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.6f}'.format)
display(df.head(10))

---
## Cell 12 — Print Full GPS Table

> Displays all 69 rows so you can inspect every point.

In [ ]:
pd.set_option('display.max_rows', 100)
display(df[['second', 'latitude', 'longitude', 'datetime_IST', 'cumulative_dist_m']])

---
## Cell 13 — Plot Lat & Lon Over Time

> Two line charts showing how latitude and longitude change across the 69-second track.  
> - **Latitude rising** → vehicle moving north  
> - **Longitude falling** → vehicle moving slightly west

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
fig.patch.set_facecolor('#0d0f12')

for ax in axes:
    ax.set_facecolor('#13161b')
    ax.tick_params(colors='#6b7585')
    ax.spines[:].set_color('#252a33')

# Latitude
axes[0].plot(df['second'], df['latitude'], color='#3fd4a0', linewidth=1.8, marker='o',
             markersize=3, markerfacecolor='#3fd4a0')
axes[0].fill_between(df['second'], df['latitude'], df['latitude'].min(),
                     alpha=0.12, color='#3fd4a0')
axes[0].set_ylabel('Latitude (°N)', color='#6b7585', fontsize=10)
axes[0].set_title('GPS Track — Latitude & Longitude over Time',
                   color='white', fontsize=12, pad=10)
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
axes[0].grid(True, color='#252a33', linewidth=0.5)

# Longitude
axes[1].plot(df['second'], df['longitude'], color='#4fc6f0', linewidth=1.8, marker='o',
             markersize=3, markerfacecolor='#4fc6f0')
axes[1].fill_between(df['second'], df['longitude'], df['longitude'].min(),
                     alpha=0.12, color='#4fc6f0')
axes[1].set_ylabel('Longitude (°E)', color='#6b7585', fontsize=10)
axes[1].set_xlabel('Time (seconds)', color='#6b7585', fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
axes[1].grid(True, color='#252a33', linewidth=0.5)

plt.tight_layout()
plt.savefig('gps_timeseries.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0f12')
plt.show()
print('✅ Saved: gps_timeseries.png')

---
## Cell 14 — Plot Cumulative Distance Over Time

> Shows total distance travelled as a function of time.  
> A straight line = constant speed. Flat sections = stationary. Steep sections = fast movement.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor('#0d0f12')
ax.set_facecolor('#13161b')
ax.tick_params(colors='#6b7585')
ax.spines[:].set_color('#252a33')

ax.plot(df['second'], df['cumulative_dist_m'],
        color='#f7c948', linewidth=2)
ax.fill_between(df['second'], df['cumulative_dist_m'],
                alpha=0.15, color='#f7c948')

# Annotate final distance
ax.annotate(f"{df['cumulative_dist_m'].iloc[-1]:.1f} m",
            xy=(df['second'].iloc[-1], df['cumulative_dist_m'].iloc[-1]),
            xytext=(-40, 12), textcoords='offset points',
            color='#f7c948', fontsize=10,
            arrowprops=dict(arrowstyle='->', color='#f7c948', lw=1))

ax.set_xlabel('Time (seconds)', color='#6b7585', fontsize=10)
ax.set_ylabel('Distance (m)', color='#6b7585', fontsize=10)
ax.set_title('Cumulative Distance Travelled', color='white', fontsize=12, pad=10)
ax.grid(True, color='#252a33', linewidth=0.5)

plt.tight_layout()
plt.savefig('gps_distance.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0f12')
plt.show()
print('✅ Saved: gps_distance.png')

---
## Cell 15 — Interactive Map with Folium

> Creates a zoomable, clickable map using **OpenStreetMap** tiles.  
> - 🟢 Green marker = start point  
> - 🔴 Red marker = end point  
> - Blue line = full route  
> - Blue dots = individual GPS points (click for popup with coordinates + time)

The map is saved as `gps_map.html` — open it in any browser for full interactivity.

In [ ]:
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=16,
    tiles='OpenStreetMap'
)

# ── Route polyline
coords = list(zip(df['latitude'], df['longitude']))
folium.PolyLine(
    coords,
    color='#3fd4a0',
    weight=4,
    opacity=0.85,
    tooltip='GPS Track'
).add_to(m)

# ── Individual point markers
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color='#4fc6f0',
        weight=1.5,
        fill=True,
        fill_color='#0d0f12',
        fill_opacity=0.7,
        tooltip=(
            f"t={int(row['second'])}s | "
            f"Lat {row['latitude']}, Lon {row['longitude']} | "
            f"{row['datetime_IST']}"
        )
    ).add_to(m)

# ── Start marker (green)
folium.Marker(
    location=[df['latitude'].iloc[0], df['longitude'].iloc[0]],
    popup=folium.Popup(
        f"<b>START</b><br>Lat: {df['latitude'].iloc[0]}<br>"
        f"Lon: {df['longitude'].iloc[0]}<br>{df['datetime_IST'].iloc[0]}",
        max_width=200
    ),
    icon=folium.Icon(color='green', icon='play', prefix='fa')
).add_to(m)

# ── End marker (red)
folium.Marker(
    location=[df['latitude'].iloc[-1], df['longitude'].iloc[-1]],
    popup=folium.Popup(
        f"<b>END</b><br>Lat: {df['latitude'].iloc[-1]}<br>"
        f"Lon: {df['longitude'].iloc[-1]}<br>{df['datetime_IST'].iloc[-1]}",
        max_width=200
    ),
    icon=folium.Icon(color='red', icon='stop', prefix='fa')
).add_to(m)

# ── Save
m.save('gps_map.html')
print('✅ Saved: gps_map.html')
print('   Open in browser for full interactivity')

# Display inline in Jupyter
display(IFrame('gps_map.html', width='100%', height=500))

---
## Cell 16 — Summary Report

> Prints a clean final summary of everything extracted.

In [ ]:
total_m  = df['cumulative_dist_m'].iloc[-1]
avg_kmh  = (total_m / 1000 / len(df)) * 3600
ok_count = df_raw['latitude'].notna().sum()

print('=' * 50)
print('  GPS EXTRACTION SUMMARY')
print('=' * 50)
print(f'  Video          : {VIDEO_PATH}')
print(f'  Location       : Lucknow, UP, India')
print(f'  Date           : 20 May 2026')
print(f'  Start time     : {df["datetime_IST"].iloc[0]}')
print(f'  End time       : {df["datetime_IST"].iloc[-1]}')
print('-' * 50)
print(f'  Total points   : {len(df)}')
print(f'  OCR success    : {ok_count}/{len(df)} ({ok_count/len(df)*100:.0f}%)')
print(f'  Interpolated   : {len(df) - ok_count}')
print('-' * 50)
print(f'  Start lat/lon  : {df["latitude"].iloc[0]}, {df["longitude"].iloc[0]}')
print(f'  End   lat/lon  : {df["latitude"].iloc[-1]}, {df["longitude"].iloc[-1]}')
print(f'  Total distance : {total_m:.1f} m  ({total_m/1000:.3f} km)')
print(f'  Avg speed      : {avg_kmh:.1f} km/h')
print('=' * 50)
print()
print('Output files:')
for f in [OUTPUT_CSV, 'gps_map.html', 'gps_timeseries.png', 'gps_distance.png']:
    exists = '✅' if os.path.exists(f) else '❌'
    print(f'  {exists} {f}')

---
## Cell 17 — Cleanup (Optional)

> Deletes the temporary `frames_temp/` folder to free disk space.  
> Run this only after you're satisfied with the results.

In [ ]:
import shutil

# Uncomment the line below to delete temporary frames
# shutil.rmtree(FRAMES_DIR)

print(f'ℹ️  Temporary frames folder: {FRAMES_DIR}/')
frame_count = len([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
folder_size = sum(os.path.getsize(os.path.join(FRAMES_DIR, f))
                  for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')) / 1024
print(f'   Contains {frame_count} frames ({folder_size:.0f} KB)')
print()
print('To delete, uncomment the shutil.rmtree line above and re-run.')